In [ ]:
# import numpy as np
# import pandas as pd
# import string
# from sklearn.preprocessing import MinMaxScaler
# from sklearn.feature_extraction.text import TfidfVectorizer
# from nltk.tokenize import word_tokenize
# from nltk.stem import WordNetLemmatizer
# from nltk.corpus import stopwords
# import nltk

# # Download necessary NLTK data
# nltk.download('punkt')
# nltk.download('punkt_tab')

# nltk.download('wordnet')
# nltk.download('stopwords')

# # Load data
# df = pd.read_csv('./imdb_movies.csv')

# # Drop non-ASCII language rows
# df = df[df['orig_lang'].apply(lambda x: x.isascii())]

# # Drop duplicates
# df = df.drop_duplicates(subset=['names', 'date_x'])

# # Convert release date
# df['release_date'] = pd.to_datetime(df['date_x'], errors='coerce')

# # Normalize budget
# df['budget_x'] = pd.to_numeric(df['budget_x'], errors='coerce')
# scaler = MinMaxScaler()
# df['budget_normalized'] = scaler.fit_transform(df[['budget_x']])

# # Split genres
# df['genre_list'] = df['genre'].fillna('').apply(lambda x: [g.strip() for g in x.split(',')])

# # Bin score into rating classes
# def map_rating(score):
#     if pd.isna(score): return np.nan
#     score = float(score)
#     if score < 30: return 0  # Very Bad
#     elif score < 50: return 1  # Bad
#     elif score < 70: return 2  # Satisfactory
#     elif score < 85: return 3  # Good
#     else: return 4  # Very Good

# df['rating_class'] = df['score'].apply(map_rating)

# # Lemmatization + cleaning
# lemmatizer = WordNetLemmatizer()
# stop_words = set(stopwords.words('english'))

# lemmatizer = WordNetLemmatizer()
# stop_words = set(stopwords.words('english'))

# def clean_text(text: str):
#     # try:
#         if not isinstance(text, str):
#             return ""
#         text = text.lower()
#         text = ''.join([char for char in text if char.isascii() and char not in string.punctuation])
#         tokens = word_tokenize(text)
#         lemmatized = [
#             lemmatizer.lemmatize(word)
#             for word in tokens
#             if word.isalpha() and word not in stop_words
#         ]
#         return " ".join(lemmatized)
#     # except Exception as e:
#     #     print("Error cleaning text:", e)
#     #     return ""

# df['clean_overview'] = df['overview'].apply(clean_text)

# # TF-IDF Vectorization
# tfidf = TfidfVectorizer(max_features=5000)
# X_tfidf = tfidf.fit_transform(df['clean_overview'])

# # now the data frame with the vectorized clean overview should be passed onto the dataframe do it here:
# # Output shapes for verification
# df.shape, X_tfidf.shape

# df


# Re-imports due to code execution reset
import numpy as np
import pandas as pd
import string
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from scipy.sparse import hstack, csr_matrix
import nltk

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

# Reload dataset
df = pd.read_csv('./imdb_movies.csv')

# Drop non-ASCII language rows
df = df[df['orig_lang'].apply(lambda x: x.isascii())]

# Drop duplicates
df = df.drop_duplicates(subset=['names', 'date_x'])

# Convert release date
df['release_date'] = pd.to_datetime(df['date_x'], errors='coerce')

# Normalize budget
df['budget_x'] = pd.to_numeric(df['budget_x'], errors='coerce')
df['budget_normalized'] = MinMaxScaler().fit_transform(df[['budget_x']].fillna(0))

# Split genres
df['genre_list'] = df['genre'].fillna('').apply(lambda x: [g.strip() for g in x.split(',')])

# Bin score into rating classes
def map_rating(score):
    if pd.isna(score): return np.nan
    score = float(score)
    if score < 30: return 0
    elif score < 50: return 1
    elif score < 70: return 2
    elif score < 85: return 3
    else: return 4

df['rating_class'] = df['score'].apply(map_rating)

# Clean and lemmatize overview
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text: str):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = ''.join([char for char in text if char.isascii() and char not in string.punctuation])
    tokens = word_tokenize(text)
    lemmatized = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    return " ".join(lemmatized)

df['clean_overview'] = df['overview'].apply(clean_text)

# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_overview'])

# Genre one-hot encoding
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])
genre_sparse = csr_matrix(genre_matrix)

# Normalize revenue
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce')
df['revenue_normalized'] = MinMaxScaler().fit_transform(df[['revenue']].fillna(0))

# Extract release year and month
df['release_year'] = df['release_date'].dt.year.fillna(0).astype(int)
df['release_month'] = df['release_date'].dt.month.fillna(0).astype(int)
year_encoded = pd.get_dummies(df['release_year'], prefix='year')
month_encoded = pd.get_dummies(df['release_month'], prefix='month')

# One-hot encode original language
language_encoded = pd.get_dummies(df['orig_lang'], prefix='lang')

# Convert additional features to sparse
budget_sparse = csr_matrix(df[['budget_normalized']].fillna(0).values)
revenue_sparse = csr_matrix(df[['revenue_normalized']].fillna(0).values)
year_sparse = csr_matrix(year_encoded.values)
month_sparse = csr_matrix(month_encoded.values)
lang_sparse = csr_matrix(language_encoded.values)

valid_rows = df['rating_class'].notna().to_numpy()



# Recreate genre_matrix with MultiLabelBinarizer (again to ensure alignment)
mlb = MultiLabelBinarizer()
df['genre_list'] = df['genre'].fillna('').apply(lambda x: [g.strip() for g in x.split(',')])
genre_matrix = mlb.fit_transform(df['genre_list'])

# Filter to valid rows
valid_rows = df['rating_class'].notna().to_numpy()

# Base human-readable columns
display_df = df.loc[valid_rows, [
    'names', 'date_x', 'score', 'rating_class', 'genre_list', 'budget_x', 'revenue', 'orig_lang'
]].reset_index(drop=True)



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\russm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\russm\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\russm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 284511 stored elements and shape (9998, 5186)>

In [ ]:
import numpy as np
import pandas as pd
import string
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from scipy.sparse import hstack, csr_matrix
import nltk
from sklearn.preprocessing import MultiLabelBinarizer


# Downloads (comment out if already done)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

# Load dataset
df = pd.read_csv('./imdb_movies.csv')

# Drop non-ASCII languages elements 
df = df[df['orig_lang'].apply(lambda x: x.isascii())]

# Drop duplicates rows 
df = df.drop_duplicates(subset=['names', 'date_x'])

# Convert date and extract year
df['release_date'] = pd.to_datetime(df['date_x'], errors='coerce')
df['release_year'] = df['release_date'].dt.year.fillna(0).astype(int)

# Normalize numeric features
df['budget_x'] = pd.to_numeric(df['budget_x'], errors='coerce')
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce')
df['budget_normalized'] = MinMaxScaler().fit_transform(df[['budget_x']].fillna(0))
df['revenue_normalized'] = MinMaxScaler().fit_transform(df[['revenue']].fillna(0))

# Process genre list and convert the comma delimited genres into an array of words 
df['genre_list'] = df['genre'].fillna('').apply(lambda x: [g.strip() for g in x.split(',')])

# Bin the rating into classes discretization
def map_rating(score):
    if pd.isna(score): return np.nan
    score = float(score)
    if score < 30: return 0
    elif score < 50: return 1
    elif score < 70: return 2
    elif score < 85: return 3
    else: return 4

# apply the discretization to the dataframe
df['rating_class'] = df['score'].apply(map_rating)

# Clean and lemmatize overview text
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


# cleans up the words for the movie overview descripton and lemmatizes it for more efficient processing 
def clean_text(text: str):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = ''.join([char for char in text if char.isascii() and char not in string.punctuation])
    tokens = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens if word.isalpha() and word not in stop_words]
    return " ".join(lemmatized)

df['clean_overview'] = df['overview'].apply(clean_text)

 # vectorize the description
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_overview'])

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])
genre_sparse = csr_matrix(genre_matrix)


budget_sparse = csr_matrix(df[['budget_normalized']].fillna(0).values)
revenue_sparse = csr_matrix(df[['revenue_normalized']].fillna(0).values)

df['release_date'] = pd.to_datetime(df['date_x'], errors='coerce')
df['release_year'] = df['release_date'].dt.year.fillna(0).astype(int)

year_encoded = pd.get_dummies(df['release_year'], prefix='year')
year_sparse = csr_matrix(year_encoded.values)


# Filter rows with valid labels
valid_rows = df['rating_class'].notna().to_numpy()
df_valid = df.loc[valid_rows].reset_index(drop=True)

# 1. Filter each feature matrix
X_tfidf_valid = X_tfidf[valid_rows]
genre_sparse_valid = genre_sparse[valid_rows]
budget_sparse_valid = budget_sparse[valid_rows]
revenue_sparse_valid = revenue_sparse[valid_rows]
lang_sparse_valid = lang_sparse[valid_rows]
year_sparse_valid = year_sparse[valid_rows]
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 2. Stack all features into one matrix
X = hstack([
    X_tfidf_valid,
    genre_sparse_valid,
    budget_sparse_valid,
    revenue_sparse_valid,
    lang_sparse_valid,
    year_sparse_valid
])

# 3. Define target variable
y = df_valid['rating_class']

# 4. Split data — capture indices so we can map predictions back
all_indices = np.arange(X.shape[0])
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X, y, all_indices, test_size=0.2, random_state=42
)

# 5. Train Naive Bayes model
model = MultinomialNB()
model.fit(X_train, y_train)

# 6. Predict
y_pred = model.predict(X_test)

# 7. Map back prediction to original movie
i = 10  # Choose test row to inspect
original_index = test_idx[i]
movie = df_valid.iloc[original_index]

print("🎬 Movie Name:", movie['names'])
print("📅 Release Date:", movie['date_x'])
print("🎭 Genres:", movie['genre_list'])
print("💰 Budget:", movie['budget_x'])
print("💸 Revenue:", movie['revenue'])
print("📝 Overview:", movie['overview'][:200], "...")
print("✅ Actual Rating Class:", y_test.iloc[i])
print("🤖 Predicted Rating Class:", y_pred[i])

# 8. Full performance report
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

# X_tfidf_valid = X_tfidf[valid_rows]
# genre_sparse_valid = genre_sparse[valid_rows]
# budget_sparse_valid = budget_sparse[valid_rows]
# revenue_sparse_valid = revenue_sparse[valid_rows]
# lang_sparse_valid = lang_sparse[valid_rows]
# year_sparse_valid = year_sparse[valid_rows]

# # Stack them
# X = hstack([
#     X_tfidf_valid,
#     genre_sparse_valid,
#     budget_sparse_valid,
#     revenue_sparse_valid,
#     lang_sparse_valid,
#     year_sparse_valid
# ])
# from sklearn.naive_bayes import MultinomialNB
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report

# # 3. Define the target variable
# y = df.loc[valid_rows, 'rating_class']

# # 4. Train-test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # 5. Initialize and train Naive Bayes model
# model = MultinomialNB()
# model.fit(X_train, y_train)

# # 6. Predict and evaluate
# y_pred = model.predict(X_test)

# print(df)

# print("Predicted Rating Class:", y_pred[3])
# print("Actual Rating Class:", y_test.iloc[3])


# print(classification_report(y_test, y_pred))

# X

# from scipy.sparse import hstack
# X = hstack([
#     X_tfidf,           # overview text
#     genre_sparse,      # genre
#     budget_sparse,     # budget
#     revenue_sparse,    # revenue
#     lang_sparse,       # original language
#     year_sparse
# ])

# X
# df

# X_tfidf_valid = X_tfidf[valid_rows]
# genre_sparse_valid = genre_sparse[valid_rows]
# y = df.loc[valid_rows, 'rating_class']

# # ✅ Step 4: Stack features horizontally
# X_stacked = hstack([
#     X_tfidf_valid,       # Overview (TF-IDF)
#     genre_sparse_valid   # Genre (one-hot)
# ])

# # ✅ Step 5: Train/test split and model
# X_train, X_test, y_train, y_test = train_test_split(X_stacked, y, test_size=0.2, random_state=42)

# model = MultinomialNB()
# model.fit(X_train, y_train)

# df

# ✅ At this point your DataFrame is fully preprocessed
# What’s ready:
# - `clean_overview`: ready for TF-IDF
# - `genre_list`: ready for MultiLabelBinarizer
# - `budget_normalized` and `revenue_normalized`: ready
# - `orig_lang`: ready for pd.get_dummies()
# - `release_year`: ready for pd.get_dummies()
# - `rating_class`: ready as your target variable



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\russm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\russm\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\russm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


🎬 Movie Name: Rings
📅 Release Date: 05/31/2017 
🎭 Genres: ['Horror']
💰 Budget: 25000000.0
💸 Revenue: 82917283.0
📝 Overview: Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her bo ...
✅ Actual Rating Class: 2
🤖 Predicted Rating Class: 2

📊 Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.21      0.33        42
           1       0.00      0.00      0.00       107
           2       0.67      0.86      0.75      1209
           3       0.57      0.39      0.46       632
           4       0.00      0.00      0.00        10

    accuracy                           0.65      2000
   macro avg       0.39      0.29      0.31      2000
weighted avg       0.60      0.65      0.61      2000



c:\Users\russm\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\russm\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\russm\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [ ]:
import numpy as np
import pandas as pd
import string
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from scipy.sparse import hstack, csr_matrix
import nltk
from sklearn.preprocessing import MultiLabelBinarizer


# Downloads (comment out if already done)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

# Load dataset
df = pd.read_csv('./imdb_movies.csv')

# Drop non-ASCII languages elements 
df = df[df['orig_lang'].apply(lambda x: x.isascii())]

# Drop duplicates rows 
df = df.drop_duplicates(subset=['names', 'date_x'])

# Convert date and extract year
df['release_date'] = pd.to_datetime(df['date_x'], errors='coerce')
df['release_year'] = df['release_date'].dt.year.fillna(0).astype(int)

# Normalize numeric features
df['budget_x'] = pd.to_numeric(df['budget_x'], errors='coerce')
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce')
df['budget_normalized'] = MinMaxScaler().fit_transform(df[['budget_x']].fillna(0))
df['revenue_normalized'] = MinMaxScaler().fit_transform(df[['revenue']].fillna(0))

# Process genre list and convert the comma delimited genres into an array of words 
df['genre_list'] = df['genre'].fillna('').apply(lambda x: [g.strip() for g in x.split(',')])

# Bin the rating into classes discretization
def map_rating(score):
    if pd.isna(score): return np.nan
    score = float(score)
    if score < 30: return 0
    elif score < 50: return 1
    elif score < 70: return 2
    elif score < 85: return 3
    else: return 4

# apply the discretization to the dataframe
df['rating_class'] = df['score'].apply(map_rating)

# Clean and lemmatize overview text
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


# cleans up the words for the movie overview descripton and lemmatizes it for more efficient processing 
def clean_text(text: str):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = ''.join([char for char in text if char.isascii() and char not in string.punctuation])
    tokens = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens if word.isalpha() and word not in stop_words]
    return " ".join(lemmatized)

df['clean_overview'] = df['overview'].apply(clean_text)

 # Vectorize the movie description using TF-IDF; turns text like "hero saves world" => [0.21, 0.00, 0.45, ...] across 5000 word features
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_overview'])


# One-hot encode the genre list so each genre becomes a binary column
# e.g., ['Action', 'Comedy'] => [1, 0, 1, 0] for ['Action', 'Adventure', 'Comedy', 'Drama']
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])
genre_sparse = csr_matrix(genre_matrix)

# Convert normalized budget and revenue into sparse format for efficient stacking
# e.g., budget 0.45 => [0.45], revenue 0.88 => [0.88]
budget_sparse = csr_matrix(df[['budget_normalized']].fillna(0).values)
revenue_sparse = csr_matrix(df[['revenue_normalized']].fillna(0).values)


# Extract release year from date and one-hot encode it
# e.g., year 2020 => [0, 0, 1, 0] for ['year_2018', 'year_2019', 'year_2020', 'year_2021']

df['release_date'] = pd.to_datetime(df['date_x'], errors='coerce')
df['release_year'] = df['release_date'].dt.year.fillna(0).astype(int)

year_encoded = pd.get_dummies(df['release_year'], prefix='year')
year_sparse = csr_matrix(year_encoded.values)


# Filter rows with valid labels
valid_rows = df['rating_class'].notna().to_numpy()
df_valid = df.loc[valid_rows].reset_index(drop=True)

# Filter out rows with missing target labels so model only trains on complete data
# Then apply the same mask to each feature matrix to keep everything aligned row-wise
X_tfidf_valid = X_tfidf[valid_rows]
genre_sparse_valid = genre_sparse[valid_rows]
budget_sparse_valid = budget_sparse[valid_rows]
revenue_sparse_valid = revenue_sparse[valid_rows]
lang_sparse_valid = lang_sparse[valid_rows]
year_sparse_valid = year_sparse[valid_rows]
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 2. Stack all features into one matrix
X = hstack([
    X_tfidf_valid,
    genre_sparse_valid,
    budget_sparse_valid,
    revenue_sparse_valid,
    lang_sparse_valid,
    year_sparse_valid
])

# 3. Define target variable
y = df_valid['rating_class']

# 4. Split data — capture indices so we can map predictions back
all_indices = np.arange(X.shape[0])
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X, y, all_indices, test_size=0.2, random_state=42
)

# 5. Train Naive Bayes model
model = MultinomialNB()
model.fit(X_train, y_train)

# 6. Predict
y_pred = model.predict(X_test)

# 7. Map back prediction to original movie
# for debugging to see how the model is predicting things
i = 10  # Choose test row to inspect
original_index = test_idx[i]
movie = df_valid.iloc[original_index]

print("🎬 Movie Name:", movie['names'])
print("📅 Release Date:", movie['date_x'])
print("🎭 Genres:", movie['genre_list'])
print("💰 Budget:", movie['budget_x'])
print("💸 Revenue:", movie['revenue'])
print("📝 Overview:", movie['overview'][:200], "...")
print("✅ Actual Rating Class:", y_test.iloc[i])
print("🤖 Predicted Rating Class:", y_pred[i])

# 8. Full performance report
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))
